In [1]:
import pandas as pd
import numpy as np


np.random.seed(42)
n_rows = 15
df = pd.DataFrame({
    'Timestamp': pd.date_range('2023-01-01', periods=n_rows, freq='min'),
    'User_ID': np.random.choice(['User_A', 'User_B'], n_rows),
    'Latency': np.random.randint(100, 300, n_rows),
    'Success': np.random.choice([True, False], n_rows, p=[0.9, 0.1])
})

df.loc[10, 'Latency'] = 800 


df['Rolling_Baseline'] = (
    df.groupby('User_ID')['Latency']
    .transform(lambda x: x.rolling(window=3, min_periods=3).mean().shift(1))
)

df['Is_Surge'] = df['Latency'] > (1.5 * df['Rolling_Baseline'])
df['Is_Surge'] = df['Is_Surge'].fillna(False)


analysis = df.groupby('Is_Surge')['Success'].value_counts(normalize=True).unstack()


print("--- Processed Data (First 12 rows) ---")
print(df[['User_ID', 'Latency', 'Rolling_Baseline', 'Is_Surge']].head(12))

print("\n--- Failure Rate Analysis ---")
print(analysis)